
**Prospective data is hard to get and we often use a hold-out dataset that was not used in development of the model for the purpose of final model performance assessment. Cross validation can be used to find the best model parameters in training.**

Here is a workflow from the Scikitlearn documentation for cross validation. Note the Test Data on the right takes no part in model development and validation on the left.

https://scikit-learn.org/stable/modules/cross_validation.html


<img src="https://scikit-learn.org/stable/_images/grid_search_workflow.png" width="400">

In [9]:
import pandas as pd
data = pd.read_csv("https://raw.githubusercontent.com/valeriesutanta/pcol3911/refs/heads/main/organophosphate_fp.csv")

In [10]:
data.shape

(278, 2049)

In [11]:
#Load data from pandas dataframe
X = data.iloc[:,:1024]
y = data['Expr'].values

In [12]:
import numpy as np
from sklearn.model_selection import train_test_split

# Define xtrain and ytrain, which are used in the assertions below.
xtrain, xtest, ytrain, ytest = train_test_split(X, y, test_size=0.15, random_state=42)

assert np.all(np.isfinite(xtrain)), "xtrain contains non-finite values."
assert np.all(np.isfinite(ytrain)), "ytrain contains non-finite values."

In [13]:
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Split the data
xtrain, xtest, ytrain, ytest = train_test_split(X, y, test_size=0.15, random_state=42)

# Define the model
rfr = RandomForestRegressor(random_state=42)

# Define the parameter grid for tuning
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': [None, 'sqrt', 'log2']
}


# Set up GridSearchCV with 5-fold cross-validation
grid_search = GridSearchCV(
    estimator=rfr,
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_squared_error',  # Metric for scoring (can use 'r2' or others)
    n_jobs=-1,  # Use all available processors
    verbose=2   # For detailed output
)

# Perform grid search
grid_search.fit(xtrain, ytrain)

# Get the best parameters and model
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

# Evaluate the best model on the test set
ypred_test = best_model.predict(xtest)
ypred_train = best_model.predict(xtrain)

# Metrics
train_mse = mean_squared_error(ytrain, ypred_train)
train_rmse = train_mse**(1/2.0)
test_mse = mean_squared_error(ytest, ypred_test)
test_rmse = test_mse**(1/2.0)

test_r2 = r2_score(ytest, ypred_test)

print("Best Parameters:", best_params)
print("Train RMSE:", train_rmse)
print("Test RMSE:", test_rmse)
print("Test R²:", test_r2)


Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best Parameters: {'max_depth': None, 'max_features': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
Train RMSE: 0.3044481189820936
Test RMSE: 0.775686934557955
Test R²: 0.7659560745094965


# Save best model for later testing

Best model from 5-fold cross validation can be saved for testing on prospective data collected or datasets not used in training. Below, we use xtest, ytest again because as you can see above they are not part of the 5-fold cross validation and selection of best parameters so they are in fact held out.

In [14]:
import joblib

# Save the best model to a file
joblib.dump(best_model, "best_random_forest_model.pkl")
print("Model saved as best_random_forest_model.pkl")


Model saved as best_random_forest_model.pkl


If we had additional hold-out data we could test the saved model on that as per below. Here we use the test set again for demonstration. Note the RMSE is the same as above.

In [15]:
# Load the saved model
loaded_model = joblib.load("best_random_forest_model.pkl")
print("Model loaded successfully.")

# Use the loaded model to make predictions on the holdout test set
yholdout_pred = loaded_model.predict(xtest)

# Evaluate performance
from sklearn.metrics import mean_squared_error, r2_score


# Metrics
holdout_mse = mean_squared_error(ytest, yholdout_pred)
holdout_rmse = holdout_mse**(1/2.0)
holdout_r2 = r2_score(ytest, yholdout_pred)

print("Holdout RMSE:", holdout_rmse)
print("Holdout R²:", holdout_r2)


Model loaded successfully.
Holdout RMSE: 0.775686934557955
Holdout R²: 0.7659560745094965
